<a href="https://colab.research.google.com/github/amitshara/Google-Playstore-Project/blob/main/03.Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import pytz

In [2]:
file_path = "/content/Play Store Data.csv"
df = pd.read_csv(file_path)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


In [5]:
df.shape

(10841, 13)

In [6]:
df = df.dropna()
df = df.drop_duplicates()

In [9]:
df['Installs'] = (
    df['Installs']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)

df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

In [11]:
df['Price'] = (
    df['Price']
    .astype(str)
    .str.replace('$', '', regex=False)
)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

In [12]:
def convert_size(size):
    try:
        size = str(size)

        if 'M' in size:
            return float(size.replace('M', ''))

        elif 'k' in size:
            return float(size.replace('k', '')) / 1024

        else:
            return np.nan

    except:
        return np.nan

In [13]:
df['Size_MBs'] = df['Size'].apply(convert_size)

In [14]:
def extract_android_version(version):
    try:
        version = str(version).split(' and up')[0]
        return float(version)
    except:
        return np.nan


df['Android_Version_Num'] = df['Android Ver'].apply(extract_android_version)

In [15]:
df['App_Name_Length'] = df['App'].astype(str).apply(len)
df['Revenue'] = df['Installs'] * df['Price']

In [16]:
filtered_df = df[
    (df['Installs'] >= 10000) &
    (df['Revenue'] >= 10000) &
    (df['Android_Version_Num'] > 4.0) &
    (df['Size_MBs'] > 15) &
    (df['Content Rating'] == 'Everyone') &
    (df['App_Name_Length'] <= 30)
].copy()


In [17]:
top_categories = (
    filtered_df.groupby('Category')['Installs']
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index
)

filtered_df = filtered_df[
    filtered_df['Category'].isin(top_categories)
]

In [18]:
summary_df = (
    filtered_df.groupby(['Category', 'Type'])
    .agg(
        Average_Installs=('Installs', 'mean'),
        Average_Revenue=('Revenue', 'mean')
    )
    .reset_index()
)

In [32]:
india_timezone = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(india_timezone)
current_hour = current_time.hour

# Graph visible only from 1 PM to 2 PM IST

if 13 <= current_hour < 14:

    # Create figure
    fig, ax1 = plt.subplots(figsize=(14, 7))

    # X-axis labels
    categories = summary_df['Category'] + ' - ' + summary_df['Type']

    # X positions
    x = np.arange(len(categories))
    width = 0.4

    # =========================
    # LEFT AXIS - INSTALLS
    # =========================

    bars = ax1.bar(
        x - width / 2,
        summary_df['Average_Installs'],
        width,
        label='Average Installs'
    )

    ax1.set_xlabel('Category and App Type')
    ax1.set_ylabel('Average Installs')
    ax1.set_xticks(x)
    ax1.set_xticklabels(categories, rotation=45, ha='right')

    # =========================
    # RIGHT AXIS - REVENUE
    # =========================

    ax2 = ax1.twinx()

    line = ax2.plot(
        x + width / 2,
        summary_df['Average_Revenue'],
        marker='o',
        linewidth=2,
        label='Average Revenue'
    )

    ax2.set_ylabel('Average Revenue ($)')

    # =========================
    # TITLE
    # =========================

    plt.title(
        'Average Installs and Revenue for Free vs Paid Apps\n'
        'Top 3 Categories'
    )

    # =========================
    # LEGEND
    # =========================

    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()

    ax1.legend(
        handles1 + handles2,
        labels1 + labels2,
        loc='upper left'
    )

    # =========================
    # SHOW GRAPH
    # =========================

    plt.tight_layout()
    plt.show()

else:
    print(
        "Dashboard graph is hidden. "
        "Graph is visible only between 1 PM and 2 PM IST."
    )

Dashboard graph is hidden. Graph is visible only between 1 PM and 2 PM IST.
